# EVALUACIÓN PARCIAL N°2: DESARROLLO DE UN AGENTE FUNCIONAL AUTÓNOMO
**Asignatura:** ISY0101 - Ingeniería de Soluciones con IA | **Duoc UC**  
**Integrantes:** Meritxell Arroyo y Matías Barraza  
**Organización:** MedisChil S.A. (Distribuidora y Bodega Farmacéutica)  
**Marco Normativo:** Decreto Supremo N° 466 de farmacias y droguerías  
**Stack de Ingeniería:** LangGraph, Google Gemini 3.8 Flash, Pydantic v2, FAISS, Pandas, RapidFuzz  
**Repositorio Oficial:** [github.com/LarviNa/collab_farmacia](https://github.com/LarviNa/collab_farmacia)

---  
### Propósito del Notebook
Este notebook contiene la **solución integral y autónoma** para la Evaluación Parcial N°2. A diferencia de un pipeline RAG lineal tradicional, este sistema implementa un **Agente Funcional Autónomo** construido con el patrón cíclico **ReAct en LangGraph**. Es capaz de sostener diálogos prolongados con memoria de corto plazo episódica (STM), recuperar normativas desde memoria semántica persistente (LTM), evaluar la viabilidad sanitaria del transporte y ejecutar transacciones oficiales de despacho y cuarentena preventiva bajo rigurosas salvaguardas de **Control de Acceso Basado en Roles (RBAC)** y **Claves de Idempotencia (UUID v4)**.

## Módulo 1: Instalación de Dependencias del Sistema Agéntico

In [ ]:
# =====================================================================
# 1. INSTALACIÓN DE DEPENDENCIAS (LANGGRAPH, LANGCHAIN, FAISS, GEMINI)
# =====================================================================
!pip install -qU \
    langgraph \
    langchain \
    langchain-core \
    langchain-community \
    langchain-google-genai \
    pydantic>=2.0 \
    faiss-cpu \
    sentence-transformers \
    rapidfuzz \
    pandas

print("✅ Dependencias instaladas y listas en el entorno de Google Colab.")

## Módulo 2: Configuración de Credenciales, Logging y Auditoría (Event Sourcing)

In [ ]:
# =====================================================================
# 2. CONFIGURACIÓN DE CREDENCIALES Y LOGGING AUDITABLE (EVENT SOURCING)
# =====================================================================
import os
import sys
import json
import uuid
import logging
import unicodedata
import urllib.request
from pathlib import Path
from datetime import datetime, timezone
from logging.handlers import RotatingFileHandler
from typing import Annotated, TypedDict, Type, Dict, Any, List, Optional, Set

# Carga segura de API Key desde Colab Secrets o entrada interactiva
try:
    from google.colab import userdata
    api_key = userdata.get('GOOGLE_API_KEY')
    if api_key:
        os.environ["GOOGLE_API_KEY"] = api_key
        print("🔑 Clave GOOGLE_API_KEY cargada exitosamente desde Colab Secrets.")
    else:
        raise ValueError("No encontrada en Secrets.")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("🔑 Ingresa tu GOOGLE_API_KEY de Google AI Studio: ")
    print("🔑 Variable GOOGLE_API_KEY configurada exitosamente.")

# Configuración del Logger Rotativo para trazabilidad auditable (ISO/DAMA)
def configurar_logger_agente() -> logging.Logger:
    logger = logging.getLogger("AgenteMedisChil_Colab")
    logger.setLevel(logging.INFO)
    formato = logging.Formatter("%(asctime)s | %(name)s | %(levelname)s | %(message)s")
    manejador = RotatingFileHandler("auditoria_operacional.log", maxBytes=1024 * 1024 * 5, backupCount=3, encoding="utf-8")
    manejador.setFormatter(formato)
    if not logger.handlers:
        logger.addHandler(manejador)
    return logger

logger_agente = configurar_logger_agente()
print("📋 Logger auditable 'auditoria_operacional.log' inicializado.")

## Módulo 3: Fuentes de Datos (Catálogo ISP con Búsqueda Difusa y Memoria Semántica LTM con FAISS)

In [ ]:
# =====================================================================
# 3. FUENTES DE DATOS: CATÁLOGO ISP Y MEMORIA SEMÁNTICA VECTORIAL (FAISS)
# =====================================================================
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# 3.1 Descarga automática del catálogo oficial ISP si no está presente
RUTA_CSV = Path("Productos_farmaceuticos_vigentes_venta_directa.csv")
URL_CSV_OFICIAL = "https://raw.githubusercontent.com/LarviNa/collab_farmacia/main/Productos_farmaceuticos_vigentes_venta_directa.csv"

if not RUTA_CSV.exists():
    print("⬇️ Descargando catálogo oficial ISP desde el repositorio...")
    urllib.request.urlretrieve(URL_CSV_OFICIAL, RUTA_CSV)
    print("✅ Catálogo descargado con éxito.")

RESPUESTA_SIN_REGISTRO = "No dispongo de registros oficiales en la bodega para responder a esta operación."

def normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", str(texto))
    return "".join(c for c in texto if not unicodedata.combining(c)).upper()

try:
    from rapidfuzz import fuzz
    def similitud_cadenas(a: str, b: str) -> float:
        return fuzz.ratio(a, b) / 100.0
except ImportError:
    import difflib
    def similitud_cadenas(a: str, b: str) -> float:
        return difflib.SequenceMatcher(None, a, b).ratio()

# Carga de inventario
columnas = ["N° Registro", "Nombre Producto", "Razon Social Titular", "Condicion Venta"]
df_inventario = pd.DataFrame(columns=columnas)
for cod in ("utf-8", "latin-1"):
    try:
        df_temp = pd.read_csv(RUTA_CSV, sep=";", encoding=cod)
        if all(col in df_temp.columns for col in columnas):
            df_inventario = df_temp.fillna("")
            break
    except Exception:
        continue

print(f"📦 Catálogo oficial ISP cargado: {len(df_inventario)} especialidades farmacéuticas vigentes.")

PALABRAS_IGNORADAS = {
    "PARA", "CUAL", "COMO", "DONDE", "ESTA", "ESTE", "ESTO", "TIENE",
    "SOBRE", "ENTRE", "BODEGA", "PRODUCTO", "MEDICAMENTO", "FARMACEUTICA",
    "FARMACEUTICO", "CUALES", "CUALQUIER", "DESDE", "HASTA", "OPERACION"
}

# 3.2 Manual Operativo de Bodega e Indexación Vectorial (LTM)
manual_operativo_bodega = [
    "NORMA TÉCNICA 01 (VACUNAS Y BIOLÓGICOS): Exigen almacenamiento estricto en cadena de frío entre 2°C y 8°C. Prohibido congelar. Transporte obligatorio en contenedores térmicos certificados con monitoreo de termógrafo continuo. Toda temperatura superior a 8°C exige bloqueo inmediato en cuarentena preventiva.",
    "NORMA TÉCNICA 02 (PRODUCTOS TÓPICOS): Los ungüentos, cremas y geles deben mantenerse en estanterías secas a temperatura ambiente controlada, nunca superior a 25°C. Evitar la exposición directa al sol y humedad excesiva.",
    "NORMA TÉCNICA 03 (DESPACHO Y VENTA DIRECTA): Todo producto con condición de Venta Directa debe ser verificado visualmente en su rotulación oficial de precios y protegido de la humedad antes de su carga al camión de distribución según Decreto Supremo 466."
]

print("🧠 Inicializando base vectorial FAISS con embeddings locales (all-MiniLM-L6-v2)...ready")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore_bodega = FAISS.from_texts(manual_operativo_bodega, embeddings)
retriever_normas = vectorstore_bodega.as_retriever(search_kwargs={"k": 2})
print("✅ Memoria semántica a largo plazo (LTM) lista para recuperación contextual.")

## Módulo 4: Subsistema Transaccional Seguro (RBAC + Idempotencia UUID v4)

In [ ]:
# =====================================================================
# 4. SUBSISTEMA TRANSACCIONAL CON CONTROL DE ACCESO (RBAC) E IDEMPOTENCIA
# =====================================================================

class ServidorTransaccionalMedisChil:
    """Subsistema transaccional seguro para mutaciones con efectos secundarios (escritura)."""

    def __init__(self) -> None:
        self.ruta_despachos = Path("data_registro_despachos.json")
        self.ruta_cuarentena = Path("data_bitacora_cuarentena.json")
        self.claves_idempotencia_procesadas: Set[str] = set()
        self._inicializar_archivos()

    def _inicializar_archivos(self):
        for ruta in (self.ruta_despachos, self.ruta_cuarentena):
            if not ruta.exists():
                with open(ruta, "w", encoding="utf-8") as f:
                    json.dump([], f, indent=2, ensure_ascii=False)
            else:
                try:
                    with open(ruta, "r", encoding="utf-8") as f:
                        for item in json.load(f):
                            if "clave_idempotencia" in item:
                                self.claves_idempotencia_procesadas.add(item["clave_idempotencia"])
                except Exception:
                    pass

    def autorizar_orden_despacho(
        self,
        token_rol: str,
        nombre_producto: str,
        n_registro: str,
        cantidad: int,
        destino: str,
        temperatura_transporte: float,
        clave_idempotencia: str
    ) -> Dict[str, Any]:
        # Salvaguarda 1: Control de Acceso Basado en Roles (RBAC)
        if token_rol not in ["OPERARIO_BODEGA", "SUPERVISOR_LOGISTICA", "DIRECTOR_TECNICO"]:
            logger_agente.warning(f"[RBAC Denegado] Rol no autorizado: {token_rol}")
            return {"status": "ERROR_SEGURIDAD_RBAC", "mensaje": f"Rol '{token_rol}' no posee privilegios para autorizar despachos."}

        # Salvaguarda 2: Clave de Idempotencia
        if clave_idempotencia in self.claves_idempotencia_procesadas:
            logger_agente.warning(f"[Idempotencia] Clave ya procesada: {clave_idempotencia}")
            return {"status": "ERROR_IDEMPOTENCIA", "mensaje": f"La clave de idempotencia '{clave_idempotencia}' ya fue procesada previamente."}

        # Salvaguarda 3: Registro persistente
        nueva_orden = {
            "id_despacho": f"DSP-{uuid.uuid4().hex[:8].upper()}",
            "clave_idempotencia": clave_idempotencia,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "producto": nombre_producto,
            "n_registro": n_registro,
            "cantidad_unidades": cantidad,
            "destino": destino,
            "temperatura_camion_c": temperatura_transporte,
            "autorizado_por_rol": token_rol,
            "estado": "AUTORIZADO_PARA_CARGA"
        }

        try:
            with open(self.ruta_despachos, "r+", encoding="utf-8") as f:
                ordenes = json.load(f)
                ordenes.append(nueva_orden)
                f.seek(0)
                json.dump(ordenes, f, indent=2, ensure_ascii=False)
            self.claves_idempotencia_procesadas.add(clave_idempotencia)
            logger_agente.info(f"Orden de despacho emitida: {nueva_orden['id_despacho']} ({nombre_producto})")
            return {"status": "EXITO", "orden": nueva_orden}
        except Exception as e:
            return {"status": "ERROR_ESCRITURA", "detalle": str(e)}

    def emitir_acta_cuarentena(
        self,
        token_rol: str,
        nombre_producto: str,
        n_registro: str,
        motivo_incumplimiento: str,
        temperatura_detectada: float,
        clave_idempotencia: str
    ) -> Dict[str, Any]:
        if clave_idempotencia in self.claves_idempotencia_procesadas:
            return {"status": "ERROR_IDEMPOTENCIA", "mensaje": f"La clave '{clave_idempotencia}' ya fue procesada."}

        acta = {
            "id_acta_cuarentena": f"CUAR-{uuid.uuid4().hex[:8].upper()}",
            "clave_idempotencia": clave_idempotencia,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "producto_bloqueado": nombre_producto,
            "n_registro": n_registro,
            "motivo_sanitario": motivo_incumplimiento,
            "temperatura_critica_c": temperatura_detectada,
            "emitida_por_rol": token_rol,
            "accion_preventiva": "Lote retenido preventivamente en cámara de aislamiento. Notificación al Director Técnico (DS N° 466)."
        }

        try:
            with open(self.ruta_cuarentena, "r+", encoding="utf-8") as f:
                actas = json.load(f)
                actas.append(acta)
                f.seek(0)
                json.dump(actas, f, indent=2, ensure_ascii=False)
            self.claves_idempotencia_procesadas.add(clave_idempotencia)
            logger_agente.warning(f"ACTA DE CUARENTENA EMITIDA: {acta['id_acta_cuarentena']} ({nombre_producto})")
            return {"status": "BLOQUEO_SANITARIO_REGISTRADO", "acta": acta}
        except Exception as e:
            return {"status": "ERROR_ESCRITURA", "detalle": str(e)}

servidor_transaccional = ServidorTransaccionalMedisChil()
print("🛡️ Servidor Transaccional Seguro (RBAC + Idempotencia) inicializado.")

## Módulo 5: Catálogo de Herramientas Tipadas con Pydantic v2 (BaseTool)

In [ ]:
# =====================================================================
# 5. DEFINICIÓN DE HERRAMIENTAS PERSONALIZADAS CON PYDANTIC V2 (BASETOOL)
# =====================================================================
from pydantic import BaseModel, Field
from langchain_core.tools import BaseTool

# --- Herramienta 1: Consulta Catálogo ISP con Fuzzy Matching ---
class ConsultaCatalogoInput(BaseModel):
    termino_busqueda: str = Field(..., description="Nombre comercial, principio activo o N° de registro sanitario del fármaco.")

class HerramientaConsultaCatalogoISP(BaseTool):
    name: str = "consultar_catalogo_isp"
    description: str = "Consulta oficial en los 2.428 medicamentos del ISP con tolerancia a errores tipográficos (Levenshtein)."
    args_schema: Type[BaseModel] = ConsultaCatalogoInput

    def _run(self, termino_busqueda: str) -> str:
        if df_inventario.empty:
            return json.dumps({"status": "ERROR", "mensaje": RESPUESTA_SIN_REGISTRO})

        term_norm = normalizar(termino_busqueda)
        coincidencias = df_inventario[
            df_inventario.apply(
                lambda fila: (
                    normalizar(fila["N° Registro"]) in term_norm
                    or any(
                        len(palabra) >= 4 and palabra not in PALABRAS_IGNORADAS and palabra in term_norm
                        for palabra in normalizar(fila["Nombre Producto"]).split()
                    )
                ),
                axis=1
            )
        ]

        # Búsqueda difusa de respaldo
        if coincidencias.empty:
            tokens_consulta = [p for p in term_norm.split() if len(p) >= 4 and p not in PALABRAS_IGNORADAS]
            if tokens_consulta:
                def coincide_difuso(fila):
                    reg_norm = normalizar(fila["N° Registro"])
                    for t in tokens_consulta:
                        if similitud_cadenas(t, reg_norm) >= 0.82:
                            return True
                    palabras_prod = [p for p in normalizar(fila["Nombre Producto"]).split() if len(p) >= 4 and p not in PALABRAS_IGNORADAS]
                    for t in tokens_consulta:
                        for pp in palabras_prod:
                            if similitud_cadenas(t, pp) >= 0.82:
                                return True
                    return False
                coincidencias = df_inventario[df_inventario.apply(coincide_difuso, axis=1)]

        if coincidencias.empty:
            return json.dumps({"status": "PRODUCTO_NO_REGISTRADO", "mensaje": RESPUESTA_SIN_REGISTRO})

        top = coincidencias.iloc[0]
        return json.dumps({
            "status": "ENCONTRADO",
            "n_registro": str(top["N° Registro"]),
            "nombre_producto": str(top["Nombre Producto"]),
            "titular": str(top["Razon Social Titular"]),
            "condicion_venta": str(top["Condicion Venta"])
        }, ensure_ascii=False)


# --- Herramienta 2: Consulta Normas Técnicas FAISS (LTM) ---
class ConsultaNormasInput(BaseModel):
    tipo_o_categoria_medicamento: str = Field(..., description="Categoría: 'vacuna', 'tópico', 'cadena de frío', 'despacho'.")

class HerramientaConsultaNormasBodega(BaseTool):
    name: str = "consultar_normas_bodega"
    description: str = "Recupera semánticamente las normas de manipulación, rango térmico y despacho desde FAISS."
    args_schema: Type[BaseModel] = ConsultaNormasInput

    def _run(self, tipo_o_categoria_medicamento: str) -> str:
        docs = retriever_normas.invoke(tipo_o_categoria_medicamento)
        if not docs:
            return json.dumps({"status": "SIN_NORMAS", "contenido": "No se encontraron directrices específicas."})
        return json.dumps({"status": "NORMAS_RECUPERADAS", "normas": [d.page_content for d in docs]}, ensure_ascii=False)


# --- Herramienta 3: Razonamiento Sanitario de Viabilidad de Transporte ---
class EvaluacionTransporteInput(BaseModel):
    nombre_producto: str = Field(..., description="Nombre del medicamento.")
    temperatura_camion_c: float = Field(..., description="Temperatura medida en el vehículo (°C).")
    tipo_producto: str = Field(..., description="'VACUNA_BIOLOGICO', 'TOPICO' o 'ESTANDAR'.")

class HerramientaEvaluacionTransporte(BaseTool):
    name: str = "evaluar_viabilidad_transporte"
    description: str = "Razonamiento sanitario para verificar si la temperatura respeta el rango normativo del DS 466."
    args_schema: Type[BaseModel] = EvaluacionTransporteInput

    def _run(self, nombre_producto: str, temperatura_camion_c: float, tipo_producto: str) -> str:
        tipo = tipo_producto.upper()
        if "VACUNA" in tipo or "BIOLOGICO" in tipo or "INFLUVAC" in nombre_producto.upper():
            if 2.0 <= temperatura_camion_c <= 8.0:
                return json.dumps({
                    "cumple_normativa": True,
                    "rango_exigido": "2°C a 8°C (Cadena de Frío)",
                    "temperatura_medida": temperatura_camion_c,
                    "conclusion": "Temperatura dentro de rango. Viable para carga."
                })
            else:
                return json.dumps({
                    "cumple_normativa": False,
                    "rango_exigido": "2°C a 8°C",
                    "temperatura_medida": temperatura_camion_c,
                    "desviacion_critica": f"Rotura de cadena de frío detectada: {temperatura_camion_c}°C está fuera del rango (2°C-8°C).",
                    "accion_requerida": "BLOQUEO_SANITARIO_INMEDIATO_CUARENTENA"
                })
        elif "TOPICO" in tipo or "UNGÜENTO" in nombre_producto.upper() or "CREMA" in nombre_producto.upper():
            if temperatura_camion_c <= 25.0:
                return json.dumps({
                    "cumple_normativa": True,
                    "rango_exigido": "Hasta 25°C",
                    "temperatura_medida": temperatura_camion_c,
                    "conclusion": "Temperatura ambiente controlada correcta."
                })
            else:
                return json.dumps({
                    "cumple_normativa": False,
                    "rango_exigido": "Hasta 25°C",
                    "temperatura_medida": temperatura_camion_c,
                    "desviacion_critica": f"Exceso térmico para tópicos: {temperatura_camion_c}°C supera los 25°C permitidos.",
                    "accion_requerida": "BLOQUEO_SANITARIO_INMEDIATO_CUARENTENA"
                })
        else:
            return json.dumps({"cumple_normativa": True, "conclusion": "Condiciones estándar aceptables."})


# --- Herramienta 4: Escritura - Emisión de Despacho con RBAC e Idempotencia ---
class EmisionDespachoInput(BaseModel):
    token_rol: str = Field(default="OPERARIO_BODEGA", description="Rol del usuario autenticado (RBAC).")
    nombre_producto: str = Field(..., description="Nombre oficial del medicamento.")
    n_registro: str = Field(..., description="Registro sanitario ISP.")
    cantidad: int = Field(..., gt=0, description="Cantidad a despachar (>0).")
    destino: str = Field(..., description="Destino oficial.")
    temperatura_transporte: float = Field(..., description="Temperatura del vehículo (°C).")
    clave_idempotencia: str = Field(default_factory=lambda: str(uuid.uuid4()), description="UUID v4 unívoco.")

class HerramientaEmisionDespacho(BaseTool):
    name: str = "emitir_orden_despacho"
    description: str = "Escritura transaccional segura que registra formalmente una orden de despacho autorizada."
    args_schema: Type[BaseModel] = EmisionDespachoInput

    def _run(self, token_rol: str, nombre_producto: str, n_registro: str, cantidad: int, destino: str, temperatura_transporte: float, clave_idempotencia: str) -> str:
        res = servidor_transaccional.autorizar_orden_despacho(
            token_rol, nombre_producto, n_registro, cantidad, destino, temperatura_transporte, clave_idempotencia
        )
        return json.dumps(res, ensure_ascii=False)


# --- Herramienta 5: Escritura - Bloqueo Sanitario y Cuarentena Preventiva ---
class BloqueoCuarentenaInput(BaseModel):
    token_rol: str = Field(default="OPERARIO_BODEGA", description="Rol que reporta la anomalía.")
    nombre_producto: str = Field(..., description="Nombre del medicamento con riesgo térmico.")
    n_registro: str = Field(..., description="Registro sanitario ISP.")
    motivo_incumplimiento: str = Field(..., description="Descripción del riesgo sanitario.")
    temperatura_detectada: float = Field(..., description="Temperatura crítica registrada.")
    clave_idempotencia: str = Field(default_factory=lambda: str(uuid.uuid4()), description="UUID v4 unívoco.")

class HerramientaBloqueoCuarentena(BaseTool):
    name: str = "emitir_acta_cuarentena"
    description: str = "Escritura contingente que retiene preventivamente el lote en cuarentena y notifica al Director Técnico."
    args_schema: Type[BaseModel] = BloqueoCuarentenaInput

    def _run(self, token_rol: str, nombre_producto: str, n_registro: str, motivo_incumplimiento: str, temperatura_detectada: float, clave_idempotencia: str) -> str:
        res = servidor_transaccional.emitir_acta_cuarentena(
            token_rol, nombre_producto, n_registro, motivo_incumplimiento, temperatura_detectada, clave_idempotencia
        )
        return json.dumps(res, ensure_ascii=False)


catalogo_herramientas = [
    HerramientaConsultaCatalogoISP(),
    HerramientaConsultaNormasBodega(),
    HerramientaEvaluacionTransporte(),
    HerramientaEmisionDespacho(),
    HerramientaBloqueoCuarentena()
]
mapa_herramientas: Dict[str, BaseTool] = {t.name: t for t in catalogo_herramientas}
print(f"🔧 {len(catalogo_herramientas)} herramientas operativas cargadas y acopladas a esquemas Pydantic v2.")

## Módulo 6: Estado Compartido (STM) y Grafo de Estados Cíclico en LangGraph

In [ ]:
# =====================================================================
# 6. ORQUESTADOR DE ESTADOS (LANGGRAPH) Y MEMORIA DE CORTO PLAZO (STM)
# =====================================================================
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# Estado Compartido Episódico (STM)
class EstadoAgenteBodega(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    id_sesion: str
    rol_usuario: str
    trazabilidad_eventos: List[Dict[str, Any]]

# Instanciación de Gemini 3.8 Flash con Tool Binding
llm_gemini = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0.0
).bind_tools(catalogo_herramientas)

PROMPT_SISTEMA_MEDISCHIL = SystemMessage(
    content=(
        "Eres el Agente Autónomo de Soporte y Seguridad Operacional de la bodega de MedisChil S.A. "
        "Tu misión es asistir al personal en consultas de catálogo, aplicación de normativas sanitarias "
        "y ejecución controlada de operaciones de despacho y cuarentena preventiva (DS N° 466 del ISP).\n\n"
        "REGLAS OPERATIVAS ESTRICTAS:\n"
        "1. PRIORIDAD SANITARIA: Antes de autorizar cualquier despacho, verifica que el producto figure en el catálogo ISP "
        "y que las condiciones del camión cumplan estrictamente la norma térmica (consultar_normas_bodega y evaluar_viabilidad_transporte).\n"
        "2. COMPORTAMIENTO ADAPTATIVO ANTE RIESGO: Si la temperatura del vehículo supera el límite permitido "
        "(ej. >8°C para vacunas o >25°C para tópicos), NUNCA autorices el despacho. Debes cambiar de plan autónomamente: "
        "rechazar la carga y ejecutar obligatoriamente 'emitir_acta_cuarentena' para retener el lote y alertar al Director Técnico.\n"
        "3. PRODUCTO NO REGISTRADO (GUARDRAIL): Si el fármaco no figura en el catálogo ISP, detén la operación y responde que "
        "no dispones de registros oficiales. No invoques herramientas de despacho.\n"
        "4. MEMORIA CONTINUA: Recuerda los fármacos y parámetros mencionados en turnos previos de la conversación."
    )
)

def nodo_razonador_manager(estado: EstadoAgenteBodega) -> dict:
    mensajes = [PROMPT_SISTEMA_MEDISCHIL] + estado["messages"]
    respuesta = llm_gemini.invoke(mensajes)
    logger_agente.info(f"Razonador ejecutado para sesión: {estado['id_sesion']}")
    return {"messages": [respuesta]}

def nodo_ejecucion_herramientas(estado: EstadoAgenteBodega) -> dict:
    ultimo_mensaje = estado["messages"][-1]
    nuevos_mensajes = []
    eventos = list(estado.get("trazabilidad_eventos", []))

    if hasattr(ultimo_mensaje, "tool_calls") and ultimo_mensaje.tool_calls:
        for tc in ultimo_mensaje.tool_calls:
            nombre = tc["name"]
            args = dict(tc["args"])
            call_id = tc["id"]

            if "token_rol" in args or nombre in ("emitir_orden_despacho", "emitir_acta_cuarentena"):
                args["token_rol"] = estado.get("rol_usuario", "OPERARIO_BODEGA")

            logger_agente.info(f"Invocando Tool: {nombre} | Args: {args}")

            if nombre in mapa_herramientas:
                try:
                    resultado = mapa_herramientas[nombre].invoke(args)
                except Exception as err:
                    resultado = json.dumps({"status": "ERROR_EJECUCION", "detalle": str(err)})
            else:
                resultado = json.dumps({"status": "ERROR", "mensaje": f"Herramienta '{nombre}' desconocida."})

            eventos.append({"timestamp": datetime.now(timezone.utc).isoformat(), "tool": nombre, "resultado": resultado})
            nuevos_mensajes.append(ToolMessage(content=str(resultado), tool_call_id=call_id))

    return {"messages": nuevos_mensajes, "trazabilidad_eventos": eventos}

def enrutar_flujo_agente(estado: EstadoAgenteBodega) -> str:
    ultimo_mensaje = estado["messages"][-1]
    if hasattr(ultimo_mensaje, "tool_calls") and ultimo_mensaje.tool_calls:
        return "ejecutar_herramientas"
    return END

# Ensamblado del Grafo de Estados cíclico ReAct
constructor = StateGraph(EstadoAgenteBodega)
constructor.add_node("razonador_manager", nodo_razonador_manager)
constructor.add_node("ejecutar_herramientas", nodo_ejecucion_herramientas)

constructor.add_edge(START, "razonador_manager")
constructor.add_conditional_edges("razonador_manager", enrutar_flujo_agente, {"ejecutar_herramientas": "ejecutar_herramientas", END: END})
constructor.add_edge("ejecutar_herramientas", "razonador_manager")

app_agente_medischil = constructor.compile()
print("🚀 Grafo de Estados LangGraph (app_agente_medischil) compilado y listo para ejecución.")

# Clase de Sesión Operativa con Memoria
class SesionOperativaBodega:
    def __init__(self, id_sesion: str = "SESION-COLAB-01", rol_usuario: str = "OPERARIO_BODEGA"):
        self.id_sesion = id_sesion
        self.rol_usuario = rol_usuario
        self.estado: EstadoAgenteBodega = {
            "messages": [],
            "id_sesion": id_sesion,
            "rol_usuario": rol_usuario,
            "trazabilidad_eventos": []
        }

    def enviar_mensaje(self, texto_usuario: str) -> str:
        self.estado["messages".strip()].append(HumanMessage(content=texto_usuario))
        self.estado = app_agente_medischil.invoke(self.estado)
        return self.estado["messages"][-1].content


## Módulo 7: Batería Oficial de Pruebas Automatizadas (Los 3 Escenarios de la Pauta)

In [ ]:
# =====================================================================
# 7. EJECUCIÓN DE PRUEBAS DE SOFTWARE: ESCENARIOS ADAPTATIVOS
# =====================================================================

print("=" * 80)
print("🏥 BANCO DE PRUEBAS DEL AGENTE FUNCIONAL - MEDISCHIL S.A. (EP2)")
print("=" * 80)

# --- ESCENARIO 1: Flujo Prolongado y Memoria Episódica STM (IE3, IE4, IE1) ---
print("\n" + "#" * 80)
print("🟢 ESCENARIO 1: Flujo Prolongado Multi-Turno y Memoria de Contexto")
print("#" * 80)

sesion_1 = SesionOperativaBodega(id_sesion="SESION-HOSPITAL-01", rol_usuario="OPERARIO_BODEGA")

turno_1 = "¿Cuál es el protocolo de almacenamiento y condición de venta para la vacuna INFLUVAC?"
print(f"\n👤 [OPERARIO - Turno 1]: {turno_1}")
print(f"🤖 [AGENTE MEDISCHIL]:\n{sesion_1.enviar_mensaje(turno_1)}")

turno_2 = "El camión de despacho N°4 tiene la cámara frigorífica en 4.5 °C. Emite la orden de despacho para 50 unidades de esa misma vacuna hacia el Hospital Central."
print(f"\n👤 [OPERARIO - Turno 2 (Referencia Anafórica)]: {turno_2}")
print(f"🤖 [AGENTE MEDISCHIL]:\n{sesion_1.enviar_mensaje(turno_2)}")

# --- ESCENARIO 2: Toma de Decisiones Adaptativas ante Contingencia (IE5, IE7, IE1) ---
print("\n" + "#" * 80)
print("🔴 ESCENARIO 2: Decisión Adaptativa ante Riesgo Sanitario (Rotura de Cadena de Frío)")
print("#" * 80)

sesion_2 = SesionOperativaBodega(id_sesion="SESION-EMERGENCIA-02", rol_usuario="SUPERVISOR_LOGISTICA")

caso_rotura = (
    "El transportista informa que el camión refrigerado falló y la temperatura interior está en 14 °C. "
    "Solicita autorizar el despacho urgente de 80 dosis de la vacuna INFLUVAC hacia la Clínica RedSalud."
)
print(f"\n👤 [SUPERVISOR]: {caso_rotura}")
print(f"🤖 [AGENTE MEDISCHIL]:\n{sesion_2.enviar_mensaje(caso_rotura)}")

# --- ESCENARIO 3: Guardrail Anti-Alucinación (IE1, IE5) ---
print("\n" + "#" * 80)
print("🛡️ ESCENARIO 3: Guardrail de Validación Previa ante Fármaco Inexistente")
print("#" * 80)

sesion_3 = SesionOperativaBodega(id_sesion="SESION-GUARDRAIL-03", rol_usuario="OPERARIO_BODEGA")

caso_inexistente = "Autoriza el despacho de 100 frascos de FANTASMIN 500 MG al Consultorio Maipú."
print(f"\n👤 [OPERARIO]: {caso_inexistente}")
print(f"🤖 [AGENTE MEDISCHIL]:\n{sesion_3.enviar_mensaje(caso_inexistente)}")

## Módulo 8: Inspección de Evidencia en Disco (Persistencia, RBAC e Idempotencia)

In [ ]:
# =====================================================================
# 8. INSPECCIÓN DE PERSISTENCIA Y REGISTROS DE SEGURIDAD EN DISCO
# =====================================================================
print("📋 INSPECCIÓN DE EVIDENCIA TRANSACCIONAL EN DISCO:\n")

print("📦 1. ÓRDENES DE DESPACHO AUTORIZADAS (data_registro_despachos.json):")
with open("data_registro_despachos.json", "r", encoding="utf-8") as f:
    despachos = json.load(f)
    print(json.dumps(despachos, indent=2, ensure_ascii=False))

print("\n⚠️ 2. ACTAS DE CUARENTENA PREVENTIVA EMITIDAS (data_bitacora_cuarentena.json):")
with open("data_bitacora_cuarentena.json", "r", encoding="utf-8") as f:
    cuarentenas = json.load(f)
    print(json.dumps(cuarentenas, indent=2, ensure_ascii=False))

## Módulo 9: Demostración Interactiva en Vivo (Demo para la Presentación Oral)

In [ ]:
# =====================================================================
# 9. CHAT INTERACTIVO EN VIVO CON EL AGENTE
# =====================================================================
# Puedes cambiar la consulta y el rol para probar cualquier caso en vivo ante el docente:
sesion_interactiva = SesionOperativaBodega(id_sesion="DEMO-INTERACTIVA", rol_usuario="SUPERVISOR_LOGISTICA")

consulta_en_vivo = "¿Cómo debe almacenarse y despacharse el ungüento BACITOPIC si la temperatura del camión es 22 °C?"
print(f"👤 Consulta: {consulta_en_vivo}\n")
print(f"🤖 Respuesta del Agente:\n{sesion_interactiva.enviar_mensaje(consulta_en_vivo)}")